In [ ]:
%cd /content
!rm -rf worldindex
!git clone https://github.com/RivaanRanawat/worldindex.git worldindex
%cd /content/worldindex

In [ ]:
!curl -sSL https://install.python-poetry.org | python3 -
import os
os.environ["PATH"] += ":/root/.local/bin"
!poetry install --with video

In [ ]:
%%bash

cat > colab_extract_aloha.py <<'PY'
from pathlib import Path

from extraction import ExtractionConfig, run_extraction
from ingestion.config import DatasetConfig


def main() -> None:
    config = ExtractionConfig(
        dataset_configs=[DatasetConfig.from_yaml(Path("config/datasets/aloha.yaml"))],
        model_id="facebook/vjepa2-vith-fpc64-256",
        device="cuda",
        batch_size=1,
        queue_depth=2,
        flush_size=10,
        start_method="spawn",
        clip_former_factory="tests.extraction.spawn_fakes:build_limited_real_clip_former",
        output_dir=Path("artifacts/extraction/gpu/aloha_smoke"),
        checkpoint_db=Path("state/gpu/extraction_aloha_smoke.sqlite3"),
    )

    last_clip_index = run_extraction(config)
    print({"last_clip_index": last_clip_index, "output_dir": str(config.output_dir)})


if __name__ == "__main__":
    main()
PY


In [ ]:
!WORLDINDEX_REAL_EXTRACTION_CLIP_LIMIT=20 poetry run python colab_extract_aloha.py

In [ ]:
%%bash
cd /content/worldindex

poetry run python - <<'PY'
from pathlib import Path
import numpy as np

raw_dir = Path("artifacts/extraction/gpu/aloha_smoke")
token_paths = sorted(raw_dir.glob("tokens_*.npy"))
metadata_paths = sorted(raw_dir.glob("metadata_*.parquet"))

print("token files:", [p.name for p in token_paths])
print("metadata files:", [p.name for p in metadata_paths])

total_clips = 0
for path in token_paths:
    arr = np.load(path, mmap_mode="r")
    total_clips += int(arr.shape[0])

print({"total_clips": total_clips})
PY


In [ ]:
%%bash

cd /content/worldindex

tar -czf /content/aloha_smoke_extraction.tgz \
  artifacts/extraction/gpu/aloha_smoke \
  state/gpu/extraction_aloha_smoke.sqlite3

In [ ]:
!ls -lh /content/aloha_smoke_extraction.tgz

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp /content/aloha_smoke_extraction.tgz /content/drive/MyDrive/


In [ ]:
%%bash

mkdir -p config/local

cat > config/local/compress_aloha_smoke.yaml <<'YAML'
raw_dir: artifacts/extraction/gpu/aloha_smoke
output_dir: artifacts/serving/aloha_smoke
checkpoint_db: state/compression_aloha_smoke.sqlite3
sample_size: 16384
pca_dim: 128
n_centroids: 128
n_bits: 2
random_seed: 0
YAML


In [ ]:
!cat config/local/compress_aloha_smoke.yaml

In [ ]:
!poetry run python scripts/compress.py config/local/compress_aloha_smoke.yaml

In [ ]:
%%bash
ls artifacts/serving/aloha_smoke
ls artifacts/serving/aloha_smoke/shards


In [ ]:
%%bash
cat > config/local/serve_aloha_smoke.yaml <<'YAML'
host: 0.0.0.0
port: 8000
data_dir: artifacts/serving/aloha_smoke
model_id: facebook/vjepa2-vith-fpc64-256
device: cuda
max_concurrent_queries: 1
YAML

In [ ]:
%%bash
cd /content/worldindex

if [ -f server.pid ]; then
  OLD_PID=$(cat server.pid)
  if kill -0 "$OLD_PID" 2>/dev/null; then
    kill "$OLD_PID"
    sleep 2
  fi
  rm -f server.pid
fi

nohup env PYTHONPATH=. poetry run python scripts/serve.py config/local/serve_aloha_smoke.yaml > /content/worldindex/server.log 2>&1 &
echo $! > /content/worldindex/server.pid
sleep 5

echo "server pid: $(cat /content/worldindex/server.pid)"
tail -n 40 /content/worldindex/server.log

In [ ]:
!curl -s http://127.0.0.1:8000/health

In [ ]:
%%bash
cd /content/worldindex
mkdir -p tmp
poetry run python - <<'PY'
from PIL import Image
Image.new("RGB", (256, 256), (120, 120, 120)).save("tmp/query.png")
PY


In [ ]:
%%bash
cd /content/worldindex
poetry run python - <<'PY'
import polars as pl
episode_id = pl.read_parquet("artifacts/serving/aloha_smoke/clips.parquet") \
    .get_column("episode_id") \
    .unique(maintain_order=True) \
    .to_list()[0]
print(episode_id)
PY

In [ ]:
!curl -s "http://127.0.0.1:8000/episodes/lerobot/aloha_sim_insertion_scripted_image_0" | python -m json.tool

In [ ]:
!grep -n 'episodes/' /content/worldindex/api/server.py


In [ ]:
%%bash
pkill -f "scripts/serve.py config/local/serve_aloha_smoke.yaml" || true
sleep 2
ps -ef | grep "scripts/serve.py" | grep -v grep || true

In [ ]:
%%bash
cd /content/worldindex
nohup env PYTHONPATH=. poetry run python scripts/serve.py config/local/serve_aloha_smoke.yaml > /content/worldindex/server.log 2>&1 &
echo $! > /content/worldindex/server.pid
sleep 5
cat /content/worldindex/server.pid
tail -n 50 /content/worldindex/server.log


In [ ]:
!curl -s "http://127.0.0.1:8000/episodes/lerobot/aloha_sim_insertion_scripted_image_0" | python -m json.tool

In [ ]:
%%bash
cd /content/worldindex
curl -s -F "image=@tmp/query.png" \
  "http://127.0.0.1:8000/search/image?top_k=5&coarse_k=20" | python -m json.tool


In [ ]:
%%bash
cd /content/worldindex
curl -s -F "image=@tmp/query.png" \
  -F 'bbox={"row_start":0,"row_end":3,"col_start":0,"col_end":3}' \
  "http://127.0.0.1:8000/search/spatial?top_k=5" | python -m json.tool


In [ ]:
%%bash
cd /content/worldindex
curl -s -F "state_a=@tmp/query.png" \
  -F "state_b=@tmp/query.png" \
  "http://127.0.0.1:8000/search/transition?top_k=5&max_gap_seconds=30" | python -m json.tool


In [ ]:
%%bash
cd /content/worldindex
mkdir -p tmp

curl -L "https://datasets-server.huggingface.co/assets/lerobot/aloha_sim_transfer_cube_scripted_image/--/5a682be7b4f6451ed450f84d3bbbe299557c1098/--/default/train/2/observation.images.top/image.jpg?Expires=1772477430&Signature=MeHttWx2rHmsp88DCOzuQUPuFQ8a5RkE9hcTUd46kWJqQmNF5ELJwzuj8OCMdNHnNFlB~rvizBZsnHt0QP0-k8UP27rICsCK4GwG9-4YGKUiARf9xBH1hX8gvAAE0QQuMPwoKBUJtnXg20iF6hQn8na9TugAcBGhd0bEC15B9s~szVvDcdHb5fT8SP~0ZrfA1ReOfVO2Yj97lvuXkjvJvaUMOUrkyAx-6kax6oWT1j756J9fNSzDnsYEA2WIAV6YDsKwwfoF7j8At5sKRixcpO2EWEkX6TPI2x8y5mlfpkC0Z7ztOIko2hCNF2Jqrq~aiJjX1zCZuDtO2am7QRd3~A__&Key-Pair-Id=K3EI6M078Z3AC3" \
  -o tmp/transfer_cube.jpg

ls -lh tmp/transfer_cube.jpg

In [ ]:
%%bash
cd /content/worldindex
curl -s -F "image=@tmp/transfer_cube.jpg;type=image/jpeg" \
  "http://127.0.0.1:8000/search/image?top_k=5&coarse_k=20" | python -m json.tool

In [ ]:
%%bash
cd /content/worldindex
curl -s -F "image=@tmp/transfer_cube.jpg;type=image/jpeg" \
  -F 'bbox={"row_start":4,"row_end":11,"col_start":4,"col_end":11}' \
  "http://127.0.0.1:8000/search/spatial?top_k=5" | python -m json.tool